# CNN Classification — Orthognathic Surgery: Before vs After

Binary image classification using a **Frozen ResNet-18** (feature extraction) to distinguish pre-surgery from post-surgery jaw profile images.

**Improvements over baseline:**
- Frozen backbone → only the classifier head is trained (prevents overfitting on 200 images)
- Stronger augmentation + dropout 0.6
- **5-fold stratified cross-validation** for reliable evaluation

**Pipeline:**
1. Imports & configuration
2. Data exploration
3. Dataset & augmentation
4. Frozen ResNet-18 model
5. 5-Fold cross-validation training
6. Learning curves per fold
7. Aggregated evaluation (accuracy, F1, ROC-AUC, confusion matrix)
8. Grad-CAM saliency maps
9. Single-image inference
10. Save best model

## 1. Imports & Configuration

In [ ]:
import os, random, copy, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    accuracy_score, f1_score, precision_score, recall_score
)
import pandas as pd

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR   = Path('dataset')
BEFORE_DIR = BASE_DIR / 'before'
AFTER_DIR  = BASE_DIR / 'after'

# ── Hyper-parameters ─────────────────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 16
NUM_EPOCHS   = 25
LR           = 5e-4      # higher LR is fine with frozen backbone
WEIGHT_DECAY = 1e-3
N_FOLDS      = 5

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Before : {len(list(BEFORE_DIR.glob("*.png")))} images')
print(f'After  : {len(list(AFTER_DIR.glob("*.png")))} images')

## 2. Data Exploration

In [ ]:
before_paths = sorted(BEFORE_DIR.glob('*.png'))
after_paths  = sorted(AFTER_DIR.glob('*.png'))
print(f'Before: {len(before_paths)}  |  After: {len(after_paths)}')

sample = Image.open(before_paths[0])
print(f'Image size: {sample.size}  mode: {sample.mode}')

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Sample Images — Before (top) vs After (bottom) Orthognathic Surgery',
             fontsize=14, fontweight='bold')
for i, ax in enumerate(axes[0]):
    ax.imshow(Image.open(before_paths[i]).convert('RGB'))
    ax.set_title('Before', color='steelblue'); ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(Image.open(after_paths[i]).convert('RGB'))
    ax.set_title('After', color='tomato'); ax.axis('off')
plt.tight_layout()
plt.savefig('cnn_samples.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved cnn_samples.png')

## 3. Dataset & DataLoaders

In [ ]:
class SurgeryDataset(Dataset):
    """Label 0 = before surgery, label 1 = after surgery."""
    CLASSES = ['before', 'after']

    def __init__(self, before_dir, after_dir, transform=None):
        self.transform = transform
        self.samples = (
            [(str(p), 0) for p in Path(before_dir).glob('*.png')] +
            [(str(p), 1) for p in Path(after_dir).glob('*.png')]
        )

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

    @property
    def labels(self):
        return [s[1] for s in self.samples]


# ── Transforms ───────────────────────────────────────────────────────────────
# Stronger augmentation to compensate for the small dataset
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Full dataset — splits are handled inside each CV fold
full_dataset = SurgeryDataset(BEFORE_DIR, AFTER_DIR)
all_labels   = np.array(full_dataset.labels)
all_indices  = np.arange(len(full_dataset))

print(f'Total samples : {len(full_dataset)}')
print(f'Class balance : Before={sum(l==0 for l in all_labels)}  After={sum(l==1 for l in all_labels)}')

## 4. Model — Frozen ResNet-18

The backbone (conv layers) is **fully frozen** — only the new classifier head is trained.

| | Full fine-tune (original) | Frozen backbone (new) |
|--|--|--|
| Trainable params | 11 177 025 | ~330 000 |
| Overfitting risk | Very high (200 images) | Low |
| Training speed | Slow | Fast |

In [ ]:
def build_frozen_resnet18():
    """ResNet-18 with frozen backbone. Only the head is trained."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Freeze every backbone parameter
    for param in model.parameters():
        param.requires_grad = False

    # Replace FC head — only these ~330k params will be updated
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.6),
        nn.Linear(256, 64),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(64, 1),
    )
    return model


def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


demo = build_frozen_resnet18()
print(f'Total params     : {sum(p.numel() for p in demo.parameters()):,}')
print(f'Trainable (head) : {count_trainable(demo):,}')

## 5. 5-Fold Cross-Validation Training

With only 200 images, a single train/val split gives an unreliable estimate.
Stratified 5-fold CV uses **all** images for evaluation and keeps class balance in every fold.

In [ ]:
def train_one_fold(train_idx, val_idx, fold_num):
    """Train frozen ResNet-18 on one fold, return metrics and history."""
    # Separate datasets with correct transforms per split
    train_ds = SurgeryDataset(BEFORE_DIR, AFTER_DIR, transform=train_tf)
    val_ds   = SurgeryDataset(BEFORE_DIR, AFTER_DIR, transform=val_tf)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        sampler=SubsetRandomSampler(train_idx), num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE,
        sampler=SubsetRandomSampler(val_idx), num_workers=0
    )

    model     = build_frozen_resnet18().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_acc = 0.0
    best_weights = None
    patience, no_improve = 8, 0
    train_accs, val_accs = [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        # Train
        model.train()
        t_correct = t_total = 0
        for imgs, labels in train_loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            criterion(logits, labels).backward()
            optimizer.step()
            preds      = (torch.sigmoid(logits) >= 0.5).long()
            t_correct += (preds == labels.long()).sum().item()
            t_total   += imgs.size(0)

        # Validate
        model.eval()
        v_correct = v_total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(DEVICE)
                labels = labels.float().unsqueeze(1).to(DEVICE)
                preds  = (torch.sigmoid(model(imgs)) >= 0.5).long()
                v_correct += (preds == labels.long()).sum().item()
                v_total   += imgs.size(0)

        scheduler.step()
        t_acc = t_correct / t_total
        v_acc = v_correct / v_total
        train_accs.append(t_acc); val_accs.append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_weights = copy.deepcopy(model.state_dict())
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_weights)

    # Collect val-fold predictions
    model.eval()
    all_probs, all_preds, all_labels_out = [], [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            logits = model(imgs.to(DEVICE)).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_preds.extend((probs >= 0.5).astype(int).tolist())
            all_labels_out.extend(labels.numpy().tolist())

    return {
        'model':      model,
        'best_acc':   best_val_acc,
        'train_accs': train_accs,
        'val_accs':   val_accs,
        'labels':     np.array(all_labels_out),
        'probs':      np.array(all_probs),
        'preds':      np.array(all_preds),
    }

In [ ]:
# Run 5-fold stratified cross-validation
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

print(f'Running {N_FOLDS}-fold stratified cross-validation...\n')

for fold, (train_idx, val_idx) in enumerate(skf.split(all_indices, all_labels), 1):
    print(f'Fold {fold}/{N_FOLDS}  (train={len(train_idx)}, val={len(val_idx)})')
    res = train_one_fold(train_idx, val_idx, fold)
    fold_results.append(res)

    acc = accuracy_score(res['labels'], res['preds'])
    auc = roc_auc_score(res['labels'], res['probs'])
    f1  = f1_score(res['labels'], res['preds'])
    print(f'  Val Acc={acc:.4f}  AUC={auc:.4f}  F1={f1:.4f}\n')

In [ ]:
# Aggregate metrics across all folds
cv_metrics = {
    'Accuracy':  [accuracy_score(r['labels'], r['preds'])  for r in fold_results],
    'Precision': [precision_score(r['labels'], r['preds']) for r in fold_results],
    'Recall':    [recall_score(r['labels'], r['preds'])    for r in fold_results],
    'F1':        [f1_score(r['labels'], r['preds'])        for r in fold_results],
    'ROC-AUC':   [roc_auc_score(r['labels'], r['probs'])   for r in fold_results],
}

print('=' * 58)
print('  5-Fold Cross-Validation Summary')
print('=' * 58)
for name, vals in cv_metrics.items():
    print(f'  {name:<12}  mean={np.mean(vals):.4f}  std={np.std(vals):.4f}'
          f'  folds={[round(v,3) for v in vals]}')

## 6. Learning Curves — Per Fold

In [ ]:
fig, axes = plt.subplots(1, N_FOLDS, figsize=(4 * N_FOLDS, 4), sharey=True)
fig.suptitle('Frozen ResNet-18 — Val Accuracy per Fold',
             fontsize=13, fontweight='bold')

for i, (ax, res) in enumerate(zip(axes, fold_results), 1):
    ep = range(1, len(res['train_accs']) + 1)
    ax.plot(ep, res['train_accs'], 'b-o', ms=3, lw=1.5, label='Train')
    ax.plot(ep, res['val_accs'],   'r-o', ms=3, lw=1.5, label='Val')
    best_acc = accuracy_score(res['labels'], res['preds'])
    ax.set_title(f'Fold {i}  (acc={best_acc:.3f})')
    ax.set_xlabel('Epoch')
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    if i == 1:
        ax.set_ylabel('Accuracy')
        ax.legend()

plt.tight_layout()
plt.savefig('cnn_fold_curves.png', dpi=120, bbox_inches='tight')
print('Saved cnn_fold_curves.png')
plt.show()

## 7. Evaluation — Aggregated Across Folds

In [ ]:
# Pool all out-of-fold predictions (covers the full dataset once)
oof_labels = np.concatenate([r['labels'] for r in fold_results])
oof_probs  = np.concatenate([r['probs']  for r in fold_results])
oof_preds  = np.concatenate([r['preds']  for r in fold_results])

roc_auc = roc_auc_score(oof_labels, oof_probs)

print('Out-of-Fold (OOF) Evaluation — all 200 images covered')
print('=' * 55)
print(classification_report(oof_labels, oof_preds,
                             target_names=['Before', 'After'], digits=4))
print(f'OOF ROC-AUC : {roc_auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(oof_labels, oof_preds)
ConfusionMatrixDisplay(cm, display_labels=['Before', 'After']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('OOF Confusion Matrix — Frozen ResNet-18')

# ROC curve
fpr, tpr, _ = roc_curve(oof_labels, oof_probs)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('OOF ROC Curve — Frozen ResNet-18')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_oof_eval.png', dpi=120, bbox_inches='tight')
print('Saved cnn_oof_eval.png')
plt.show()

## 8. Grad-CAM Saliency Maps

Highlights which image regions the model focuses on when classifying.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.gradients = None; self.activations = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_backward_hook(self._bwd)
    def _fwd(self, m, i, o): self.activations = o.detach()
    def _bwd(self, m, gi, go): self.gradients = go[0].detach()
    def __call__(self, tensor):
        self.model.eval()
        x = tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
        self.model(x).backward()
        w   = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.activations).sum(1, keepdim=True))
        cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def denorm(t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


# Use the best fold model for Grad-CAM
best_fold_idx = int(np.argmax([accuracy_score(r['labels'], r['preds']) for r in fold_results]))
best_model    = fold_results[best_fold_idx]['model']
print(f'Using Fold {best_fold_idx+1} model for Grad-CAM')

grad_cam = GradCAM(best_model, best_model.layer4[-1].conv2)

# Build a val loader from the best fold's val indices for sample images
skf2 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf2.split(all_indices, all_labels))
_, best_val_idx = splits[best_fold_idx]

sample_ds     = SurgeryDataset(BEFORE_DIR, AFTER_DIR, transform=val_tf)
sample_loader = DataLoader(sample_ds, batch_size=8,
                           sampler=SubsetRandomSampler(best_val_idx), num_workers=0)
sample_imgs, sample_lbls = next(iter(sample_loader))

n_show = min(6, len(sample_imgs))
sample_imgs = sample_imgs[:n_show]
sample_lbls = sample_lbls[:n_show]
CLASS_NAMES  = ['Before', 'After']

fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
fig.suptitle(f'Grad-CAM — Frozen ResNet-18 (Fold {best_fold_idx+1})\n'
             'top: original | mid: heatmap | bottom: overlay',
             fontsize=12, fontweight='bold')

for col, (img_t, lbl) in enumerate(zip(sample_imgs, sample_lbls)):
    img_np = denorm(img_t)
    cam    = grad_cam(img_t)
    axes[0, col].imshow(img_np)
    axes[0, col].set_title(f'GT: {CLASS_NAMES[lbl]}', fontsize=9)
    axes[1, col].imshow(cam, cmap='jet')
    axes[2, col].imshow(img_np)
    axes[2, col].imshow(cam, cmap='jet', alpha=0.45)
    for r in range(3): axes[r, col].axis('off')

plt.tight_layout()
plt.savefig('cnn_gradcam.png', dpi=120, bbox_inches='tight')
print('Saved cnn_gradcam.png')
plt.show()

## 9. Model Comparison — Before vs After Improvements

In [ ]:
# Original baseline results (from the first run)
baseline = {
    'Accuracy': 0.500, 'Precision': 0.500,
    'Recall': 0.455,   'F1': 0.476, 'ROC-AUC': 0.606
}
improved = {k: np.mean(v) for k, v in cv_metrics.items()}

df = pd.DataFrame({'Baseline (full fine-tune)': baseline,
                   'Improved (frozen + 5-fold)': improved}).T
print(df.round(4).to_string())

# Bar chart
x     = np.arange(len(baseline))
width = 0.35
names = list(baseline.keys())

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, [baseline[k] for k in names], width,
            label='Baseline (full fine-tune)', color='#e07b54', edgecolor='grey')
b2 = ax.bar(x + width/2, [improved[k] for k in names], width,
            label='Improved (frozen + 5-fold CV)', color='#4c9be8', edgecolor='grey')

ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0, 1.15)
ax.set_title('Baseline vs Improved — Frozen ResNet-18 + 5-Fold CV',
             fontsize=13, fontweight='bold')
ax.axhline(0.5, color='grey', linestyle='--', lw=1, label='Random baseline')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bar in b1:
    ax.annotate(f'{bar.get_height():.3f}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=8)
for bar in b2:
    ax.annotate(f'{bar.get_height():.3f}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=8,
                color='#1a5fa0', fontweight='bold')

plt.tight_layout()
plt.savefig('cnn_comparison.png', dpi=120, bbox_inches='tight')
print('Saved cnn_comparison.png')
plt.show()

## 10. Single-Image Inference

In [ ]:
def predict_image(model, image_path):
    class_names = ['Before Surgery', 'After Surgery']
    img    = Image.open(image_path).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        logit = model(tensor).squeeze().item()
    p_after  = torch.sigmoid(torch.tensor(logit)).item()
    p_before = 1 - p_after
    pred     = class_names[int(p_after >= 0.5)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img)
    ax1.set_title(f'Prediction: {pred}', fontsize=12, fontweight='bold')
    ax1.axis('off')
    bars = ax2.bar(class_names, [p_before, p_after],
                  color=['steelblue', 'tomato'], edgecolor='grey')
    ax2.set_ylim(0, 1.15); ax2.set_ylabel('Probability')
    ax2.set_title('Class Probabilities'); ax2.grid(axis='y', alpha=0.3)
    for bar, p in zip(bars, [p_before, p_after]):
        ax2.text(bar.get_x() + bar.get_width() / 2, p + 0.02,
                 f'{p:.3f}', ha='center', fontsize=11)
    plt.tight_layout(); plt.show()
    print(f'Predicted: {pred}  |  P(before)={p_before:.4f}  P(after)={p_after:.4f}')
    return pred, p_before, p_after


before_imgs = list(BEFORE_DIR.glob('*.png'))
after_imgs  = list(AFTER_DIR.glob('*.png'))
print('--- Before image ---')
predict_image(best_model, str(before_imgs[5]))
print('--- After image ---')
predict_image(best_model, str(after_imgs[5]))

## 10. Save Best Model

## 11. Upload & Test — Drag Your Own Image

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# ── Upload widget ─────────────────────────────────────────────────────────────
upload_btn = widgets.FileUpload(
    accept='.png,.jpg,.jpeg',
    multiple=False,
    description='Upload Image',
    button_style='info',
    layout=widgets.Layout(width='200px')
)

run_btn = widgets.Button(
    description='Run Prediction',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='160px')
)

out = widgets.Output()

def on_predict(b):
    with out:
        clear_output(wait=True)
        if not upload_btn.value:
            print('Please upload an image first.')
            return

        # Read uploaded bytes → PIL image
        uploaded = list(upload_btn.value.values())[0]
        img_bytes = uploaded['content']
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')

        # Inference with the best fold model
        tensor = val_tf(img).unsqueeze(0).to(DEVICE)
        best_model.eval()
        with torch.no_grad():
            logit = best_model(tensor).squeeze().item()

        p_after  = torch.sigmoid(torch.tensor(logit)).item()
        p_before = 1 - p_after
        pred     = 'After Surgery' if p_after >= 0.5 else 'Before Surgery'
        conf     = max(p_after, p_before) * 100

        # Display
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

        ax1.imshow(img)
        color = 'tomato' if p_after >= 0.5 else 'steelblue'
        ax1.set_title(f'Prediction: {pred}\nConfidence: {conf:.1f}%',
                      fontsize=13, fontweight='bold', color=color)
        ax1.axis('off')

        bars = ax2.bar(['Before Surgery', 'After Surgery'],
                       [p_before, p_after],
                       color=['steelblue', 'tomato'], edgecolor='grey')
        ax2.set_ylim(0, 1.15)
        ax2.set_ylabel('Probability', fontsize=11)
        ax2.set_title('Class Probabilities', fontsize=12)
        ax2.grid(axis='y', alpha=0.3)
        for bar, p in zip(bars, [p_before, p_after]):
            ax2.text(bar.get_x() + bar.get_width() / 2, p + 0.02,
                     f'{p:.3f}', ha='center', fontsize=12, fontweight='bold')

        # Highlight the predicted bar
        bars[int(p_after >= 0.5)].set_edgecolor('black')
        bars[int(p_after >= 0.5)].set_linewidth(2.5)

        plt.tight_layout()
        plt.show()
        print(f'Result  : {pred}')
        print(f'P(Before Surgery) = {p_before:.4f}')
        print(f'P(After  Surgery) = {p_after:.4f}')

run_btn.on_click(on_predict)

display(widgets.HBox([upload_btn, run_btn]))
display(out)

In [ ]:
torch.save({
    'model_name':  'FrozenResNet18',
    'state_dict':  best_model.state_dict(),
    'class_names': ['Before Surgery', 'After Surgery'],
    'img_size':    IMG_SIZE,
    'fold':        best_fold_idx + 1,
    'oof_auc':     roc_auc,
}, 'cnn_best_model.pth')

best_acc = accuracy_score(fold_results[best_fold_idx]['labels'],
                          fold_results[best_fold_idx]['preds'])
print(f'Best fold      : {best_fold_idx + 1}  (val acc={best_acc:.4f})')
print(f'OOF ROC-AUC    : {roc_auc:.4f}')
print(f'Mean CV Acc    : {np.mean(cv_metrics["Accuracy"]):.4f} ± {np.std(cv_metrics["Accuracy"]):.4f}')
print('Saved          : cnn_best_model.pth')